In [1]:
%pip install groq
%pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.3 MB/s eta 0:00:00


In [24]:
import os
from dotenv import load_dotenv

#Obter chave da API do groq
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
# Nessa parte foi feita uma alteração no código em relação ao vídeo, para que não fosse disponibilizada a API key

In [25]:
import os
from groq import Groq
# Configurando o modelo que usaremos para esse teste
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in natural language processing (NLP) and have numerous applications in various industries. The importance of fast language models can be understood from the following perspectives:

1. **Real-time Applications**: Fast language models enable real-time applications such as chatbots, virtual assistants, and language translation systems. These models can process and respond to user input quickly, providing a seamless and interactive experience.
2. **Efficient Processing**: Fast language models can handle large volumes of text data efficiently, reducing the computational resources and time required for processing. This is particularly important in applications where data is constantly streaming in, such as social media monitoring or news analysis.
3. **Improved User Experience**: Fast language models can significantly enhance the user experience by providing quick and accurate responses to queries. This is especially important in customer service, where time

Criando a classe Agent

In [26]:
class Agent: # Inicializamos o Agent com um cliente e um sistema
  def __init__(self, client: Groq, system: str = "") -> None:
      self.client = client
      self.system = system # Essa é a mensagem do sistema, isto é o system prompt (basicamente um roteiro de trabalho para o Agente)
      self.messages: list = []
      if self.system is not None:
          self.messages.append({"role": "system", "content": system}) # Desde que haja a mensagem do sistema, será incluida na lista de mensagens, a qual fornece as instruções iniciais para o modelo

  def __call__(self, message=""): # É acionada sempre que é feita uma chamada ao Agente
      if message:
          self.messages.append({"role": "user", "content": message})
      result = self.execute()
      self.messages.append({"role": "assistant", "content": result})
      return result

  def execute(self): # Executa uma conclusão com todo histórico de mensagem
      completion = client.chat.completions.create(
          messages=self.messages,
          model="llama-3.3-70b-versatile",
      )
      return completion.choices[0].message.content

Criando o System Prompt

In [27]:
# No System Prompt é passado uma espécie de roteiro para o Agente seguir
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944x10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944x10e25

Now it's your turn:
""".strip()


Escrevendo as funções disponíveis para o Agente, isto é as ferramentas do Agente.

In [28]:
# As ferramentas (tools) disponíveis para um Agente são essencialmente funções
def calculate(operation):
    return eval(operation)

def get_planet_mass(planet) -> float:
    match planet.lower():
      case "earth":
        return 5.972e24
      case "mars":
        return 6.39e23
      case "jupiter":
        return 1.898e27
      case "saturn":
        return 5.683e26
      case "uranus":
        return 8.681e25
      case "neptune":
        return 1.024e26
      case "mercury":
        return 3.285e23
      case "venus":
        return 4.867e24
      case _:
        return 0.0

Inicializando nosso Agente (sem loop)

In [29]:
neil_tyson = Agent(client, system_prompt)

In [30]:
result = neil_tyson("What is the mass of Earth times 5?")
print(result)

Thought: To find the mass of Earth times 5, I first need to get the mass of Earth and then multiply it by 5.
Action: get_planet_mass: Earth
PAUSE


In [31]:
neil_tyson.messages # Essa linha permite a gente ver que primeiro o agente é inicalizado com a mensagem do sistema e depois que a do usuário

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944x10e25\n\nIf you have the a

In [32]:
result = neil_tyson()
print(result)

In [33]:
observation = get_planet_mass("Earth")
print(observation)

5.972e+24


In [34]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: Now that I have the mass of Earth, I can multiply it by 5 to find the result.
Action: calculate: 5.972e24 * 5
PAUSE


In [35]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944x10e25\n\nIf you have the a

In [36]:
result = neil_tyson()
print(result)

In [37]:
observation = calculate("5.972e24 * 5")
print(observation)

2.9860000000000004e+25


In [38]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: I have now calculated the mass of Earth times 5, so I can output the result as the answer.
Answer: The mass of Earth times 5 is 2.986e+25


In [39]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944x10e25\n\nIf you have the a

Rodando agora o Agente com loop

In [40]:
import re

def agent_loop(max_iterations, system, query):
  agent = Agent(client, system_prompt) # incializa o Agente
  tools = ['calculate', 'get_planet_mass'] # inicializa os nomes das 'tools' que se tem disponível
  next_prompt = query # O primeiro prompt é definido como a pergunta que o user fizer para o Agente
  i = 0
  while i < max_iterations: # Inicializa-se o loop, com um máximo de interações que é definida na chamada da função
    i += 1
    result = agent(next_prompt) # Chama o Agente
    print(result)

    if "PAUSE" in result and "Action" in result: # Verifica o resultado obtido da chamada do Agente
      action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
      chosen_tool = action[0][0]
      arg = action[0][1]

      if chosen_tool in tools: # Verifica se a ferramenta escolhida está na lista de ferramentas disponíveis
        result_tool = eval(f"{chosen_tool}('{arg}')")
        next_prompt = f"Observation: {result_tool}"

      else:
        next_prompt = "Observation: Tool not found"

      print(next_prompt)
      continue # Serve para encerrar o primeiro 'if' do loop, para que reinicie o loop

    if "Answer" in result:
      break # Serve para avisar que terminou a tarefa e encerra o loop


In [41]:
agent_loop(max_iterations=10, system=system_prompt, query="What is the mass of the Earth plus the mass of Mercury, all of this times 5?")

Thought: To solve this problem, I need to find the mass of Earth and the mass of Mercury, then add them together, and finally multiply the result by 5. First, I'll find the mass of Earth.

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Mercury to add them together.

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: Now that I have the masses of both Earth and Mercury, I can add them together and then multiply the result by 5. First, I'll add the masses.

Action: calculate: 5.972e+24 + 3.285e+23
PAUSE
Observation: 6.300500000000001e+24
Thought: Now that I have the sum of the masses of Earth and Mercury, I can multiply this result by 5 to get the final answer.

Action: calculate: 6.300500000000001e+24 * 5
PAUSE
Observation: 3.1502500000000004e+25
Thought: I have now calculated the mass of Earth plus the mass of Mercury, all multiplied by 5, so I can output the final answer.

Answer: T